# Advanced Session: Electrolyte Conductivity Challenge
AI for Materials Science — Hands-on session 5 (optional)

A one-week, Kaggle-style challenge: predict the **ionic conductivity k (mS/cm)** of liquid electrolytes
from their salt, concentration, solvent mixture and temperature. You get a training file with labels and
a test file without them; the site scores your predictions against hidden labels.

**Site:** <SPACE_URL>

- Task: predict ionic conductivity k (mS/cm) of liquid electrolytes, CALiSol-23 test split. Metric: RMSE on the hidden labels (lower is better).
- Public leaderboard: 30% of the test formulations; private: the remaining 70%, revealed after the deadline.
- Open: 2026-11-02 17:30 KST. Deadline: 2026-11-08 23:59 KST. Private results: 2026-11-09 09:00 KST.
- Submission: a CSV with the header id,k covering every test id, plus a Colab or Google Drive link to your notebook.
- Limit: 5 accepted submissions per student per KST day. Rejected files do not count.
- Identity: your student ID and the 6-digit access code sent to you on KLMS. Only your nickname is shown.
- Final ranking uses the private RMSE of each student's best public submission.
- Do not look up or reuse the original CALiSol-23 values for the test formulations; build a model from the training file. Suspicious entries are flagged and may be disqualified.
- The top 3 notebooks are reproduced by the TA before results are announced in class on Nov 9.
- This is an optional activity; any bonus credit is decided by the instructor.
- Too many failed logins trigger a 15-minute cool-down; 30 failures in a day lock that ID for the day.
- Locked? Contact the TA or the site owner; a Space restart clears locks.

---

Data: CALiSol-23, de Blasio et al., Sci. Data 11, 750 (2024), CC BY 4.0 (modified: cleaned and split).

**First:** File ▸ Save a copy in Drive, and work in your copy.

Run the cells one at a time from the top. Later cells reuse variables created in earlier ones, so skipping ahead gives you a "name is not defined" error. Lines starting with `##` inside the code are comments written for you; Python does not run them.

**Shared or lab PC?** Sign out of Google when you finish and choose Runtime ▸ Disconnect and delete runtime.

## 0. Setup

### 0-1. Check the libraries
Nothing to install: pandas, scikit-learn, PyTorch and Matplotlib ship with Colab.

In [ ]:
import pandas, sklearn, torch
print("pandas", pandas.__version__, " scikit-learn", sklearn.__version__,
      " torch", torch.__version__)

### 0-2. Download the course material

In [ ]:
!git clone https://github.com/kwongibaek/MS49900-AI4M.git

### 0-3. Import the libraries

In [ ]:
import copy, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(0)
torch.manual_seed(0)

### 0-4. Set the file paths

In [ ]:
DATA_DIR = Path("MS49900-AI4M/Data")
OUTPUT = Path("outputs")
OUTPUT.mkdir(exist_ok=True)
print("Data folder exists:", DATA_DIR.exists())

## A. The data

### A-1. Load the training and test files
`train` has the target column `k`; `test` has an `id` column instead. Your submission maps every test `id`
to a predicted `k`.

In [ ]:
train = pd.read_csv(DATA_DIR / "calisol23_train.csv")
test = pd.read_csv(DATA_DIR / "calisol23_test.csv")
print("train:", train.shape, " test:", test.shape)
train.head()

### A-2. What the columns mean
- `T` — temperature in kelvin.
- `c` — lithium-salt concentration, in the unit given by `c units` (mol/kg or mol/L).
- `salt` — the lithium salt (LiPF6, LiBF4, LiBOB, ...).
- `solvent ratio type` — whether the solvent columns are weight (`w`), volume (`v`) or mole (`mol`) fractions.
- 38 solvent columns (EC, PC, DMC, EMC, DEC, DME, ...) — the fraction of each solvent; each row sums to 1.
- `k` — ionic conductivity in mS/cm (train only). Most values are below 15 mS/cm.

The hidden test labels are split into a public part (shown on the leaderboard during the week) and a
private part (final ranking). Every formulation appears on one side only, so the test rows are new
salt/solvent/concentration combinations, not new temperatures of a known mixture.

### A-3. The target distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.hist(train["k"], bins=60)
ax.set_xlabel("ionic conductivity k (mS/cm)"); ax.set_ylabel("rows")
ax.set_title("Training set")
fig.savefig(OUTPUT / "A3_k_hist.png", dpi=150, bbox_inches="tight")
plt.show()
print(train["k"].describe().round(3))

## B. Features
The three text columns are one-hot encoded (`handle_unknown="ignore"` so that a category seen only in test
does not crash the model); the numeric columns pass through unchanged.

In [ ]:
CATEGORICAL = ["salt", "c units", "solvent ratio type"]
NUMERIC = [c for c in train.columns if c not in CATEGORICAL + ["k"]]
features = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
                              ("num", "passthrough", NUMERIC)])
X, y = train.drop(columns="k"), train["k"]
X_test = test.drop(columns="id")
print(len(NUMERIC), "numeric columns +", len(CATEGORICAL), "categorical columns")

## C. Validate the way the leaderboard scores

### C-1. Formulations as groups
The leaderboard's hidden split keeps every row of one formulation (salt, concentration, unit, ratio type and
the 38 solvent fractions) on one side. Validate the same way, or your local score will be too optimistic.

In [ ]:
KEY = ["salt", "c", "c units", "solvent ratio type"] + [c for c in NUMERIC if c not in ("T", "c")]
groups = train[KEY].astype(str).agg("|".join, axis=1)
print(groups.nunique(), "formulations in", len(train), "training rows")

### C-2. Grouped 5-fold RMSE for three models
RMSE is the leaderboard metric. Ridge regression is a linear reference, the random forest is the baseline
that sits on the leaderboard as "Baseline (starter notebook)", and a small PyTorch MLP is the deep-learning
candidate from class. Inside every fold the one-hot encoding and the scaling are fitted on that fold's
training rows only, and the MLP is capped at 30 seconds per fold.

In [ ]:
class TorchMLP(BaseEstimator, RegressorMixin):
    """A small CPU MLP that behaves like a scikit-learn regressor inside a pipeline; deterministic seed."""
    def __init__(self, hidden=(64, 32), epochs=40, lr=1e-3, batch_size=256, max_seconds=30, seed=0):
        self.hidden, self.epochs, self.lr = hidden, epochs, lr
        self.batch_size, self.max_seconds, self.seed = batch_size, max_seconds, seed

    def _tensor(self, X):
        X = X.toarray() if hasattr(X, "toarray") else np.asarray(X)   ## one-hot output is sparse
        return torch.tensor(X, dtype=torch.float32)

    def fit(self, X, y):
        torch.manual_seed(self.seed)
        Xt = self._tensor(X)
        yt = torch.tensor(np.asarray(y, dtype=np.float32)).reshape(-1, 1)
        layers, n_in = [], Xt.shape[1]
        for h in self.hidden:
            layers += [nn.Linear(n_in, h), nn.ReLU()]
            n_in = h
        self.net_ = nn.Sequential(*layers, nn.Linear(n_in, 1))   ## trailing _ = fitted (scikit-learn convention)
        optimizer = torch.optim.Adam(self.net_.parameters(), lr=self.lr)
        loader = DataLoader(TensorDataset(Xt, yt), batch_size=self.batch_size, shuffle=True)
        start = time.time()
        for epoch in range(self.epochs):
            for xb, yb in loader:
                optimizer.zero_grad()
                loss = nn.functional.mse_loss(self.net_(xb), yb)
                loss.backward()
                optimizer.step()
            if time.time() - start > self.max_seconds:
                break
        return self

    def predict(self, X):
        self.net_.eval()
        with torch.no_grad():
            return self.net_(self._tensor(X)).numpy().ravel()

def grouped_rmse(model, X, y, groups, n_splits=5):
    scores = []
    for fit_idx, val_idx in GroupKFold(n_splits=n_splits).split(X, y, groups=groups):
        model.fit(X.iloc[fit_idx], y.iloc[fit_idx])
        pred = model.predict(X.iloc[val_idx])
        scores.append(np.sqrt(mean_squared_error(y.iloc[val_idx], pred)))
    return float(np.mean(scores)), float(np.std(scores))

candidates = {
    "Ridge": make_pipeline(copy.deepcopy(features), StandardScaler(with_mean=False), Ridge(alpha=1.0)),
    "RandomForest": make_pipeline(copy.deepcopy(features),
                                  RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=0)),
    "MLP (PyTorch)": make_pipeline(copy.deepcopy(features), StandardScaler(with_mean=False), TorchMLP()),
}
for name, model in candidates.items():
    t0 = time.time()
    mean, std = grouped_rmse(model, X, y, groups)
    print(f"{name:13s} grouped CV RMSE = {mean:.3f} +- {std:.3f}   ({time.time() - t0:.0f} s)")

## D. Make a submission

### D-1. Fit the baseline on all training rows and predict the test file
The checks at the end are the same ones the site runs: every test id exactly once, numeric and finite values.

In [ ]:
baseline = candidates["RandomForest"].fit(X, y)
submission = pd.DataFrame({"id": test["id"], "k": baseline.predict(X_test)})
assert len(submission) == len(test) and set(submission["id"]) == set(test["id"])
assert np.isfinite(submission["k"]).all()
submission.to_csv(OUTPUT / "submission.csv", index=False)
print("wrote", OUTPUT / "submission.csv", "with", len(submission), "rows")
submission.head()

### D-2. Submit
1. Run the cell below: in Colab it downloads `submission.csv` to your computer (the site's file picker reads
   your computer, not the Colab runtime). You can also download it from Colab's Files panel (`outputs/`).
2. Open the site linked at the top of this notebook.
3. Enter your student ID and the 6-digit code from KLMS, paste a share link to this notebook
   (File ▸ Share ▸ "Anyone with the link: Viewer"), and upload the downloaded `submission.csv`.
4. Read the receipt: it shows your public RMSE and how many submissions you have left today (5 per day).

The Baseline row on the leaderboard is this exact notebook, so your first job is to beat it.

In [ ]:
try:   ## in Colab: download the file to your computer, then upload it on the site
    from google.colab import files
    files.download(str(OUTPUT / "submission.csv"))
except ImportError:
    print("Not in Colab: outputs/submission.csv is in the working folder.")

## E. Ideas to try
- Predict `log(k)` instead of `k` and transform back; conductivity spans two orders of magnitude.
- Add `1/T` as a feature: conductivity follows an Arrhenius-like temperature dependence.
- Merge salt synonyms (`LiTFSI` and `LiN(CF3SO2)2` are the same salt) and group rare salts.
- Try `HistGradientBoostingRegressor`, or a small PyTorch MLP on the one-hot + scaled numeric features.
- Tune the random forest (`max_features`, `min_samples_leaf`) with the grouped CV from C-2, never with the leaderboard.

### E-2. Practice 1
Implement one idea above, measure it with `grouped_rmse`, and if it beats the baseline write a new
`outputs/submission.csv` and submit it.

In [ ]:
## Write your code here.
## Order: build a new pipeline -> grouped_rmse(...) -> compare with the baseline -> fit on all rows -> write submission.csv

## Data sources

### CALiSol-23
`Data/calisol23_train.csv`, `Data/calisol23_test.csv`, `Data/calisol23_sample_submission.csv`. P. de Blasio, J. Elsborg, T. Vegge, E. Flores, A. Bhowmik, CALiSol-23: Experimental electrolyte conductivity data for various Li-salts and solvent combinations, *Scientific Data* 11, 750 (2024), https://doi.org/10.1038/s41597-024-03575-8. Dataset: DTU Data, https://doi.org/10.11583/DTU.c.6929599.v1, licensed under CC BY 4.0 (https://creativecommons.org/licenses/by/4.0/). Changes: 556 rows removed (missing or non-positive k, negative c); whitespace stripped; doi column removed; split into train/test by formulation; test labels withheld for the competition.